In [82]:
import sys
import subprocess
import importlib
import importlib.util
import urllib.request
from pathlib import Path
import numpy as np
import json
import textwrap
from IPython.display import clear_output, display
import ipywidgets as widgets

def check_ollama_available(timeout_sec: float = 1.5) -> bool:
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=timeout_sec) as response:
            return int(getattr(response, 'status', 0)) == 200
    except Exception:
        return False

OLLAMA_AVAILABLE = check_ollama_available()
if OLLAMA_AVAILABLE:
    print('Ollama available — NLG enabled')
else:
    print('Ollama not available — set NLG_ENABLED=False in Cell 2')

def ensure_gymnasium() -> None:
    if importlib.util.find_spec('gymnasium') is None:
        print('Installing missing dependency: gymnasium')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gymnasium==1.2.1'])

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'Simulation_4' / 'env').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root containing Simulation_4/env')

ensure_gymnasium()
repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Import from concrete modules and reload so SupportEnv and NLGLayer share the same fresh objects.
import Simulation_4.env.nlg_layer as nlg_layer_mod
import Simulation_4.env.support_env as support_env_mod
importlib.reload(nlg_layer_mod)
importlib.reload(support_env_mod)
SupportEnv = support_env_mod.SupportEnv
NLGLayer = nlg_layer_mod.NLGLayer

artifacts_root = repo_root / 'Simulation_4' / 'artifacts'

Ollama available — NLG enabled


In [98]:
# Set nlg_enabled=True if Ollama is running locally, False otherwise
NLG_ENABLED = OLLAMA_AVAILABLE

# Optional: fix a subflow for testing specific scenarios
# Set to None to sample randomly
SUBFLOW_FILTER = None  # e.g. ['return_size', 'manage_dispute_bill']

# Optional: fix a seed for reproducibility
SEED = 9600  # set to integer for reproducible episodes

In [99]:
import inspect
import Simulation_4.env.support_env as support_env_mod
from Simulation_4.env.nlg_layer import NLGLayer as NLGFromModule

classes_to_patch = []
if hasattr(support_env_mod, 'NLGLayer'):
    classes_to_patch.append(support_env_mod.NLGLayer)
classes_to_patch.append(NLGFromModule)

patched_count = 0
for cls in set(classes_to_patch):
    sig = inspect.signature(cls.build_system_prompt)
    if 'identity' in sig.parameters and 'free_info' in sig.parameters:
        continue

    _orig_build_system_prompt = cls.build_system_prompt

    def _compat_build_system_prompt(
        self,
        persona_label,
        rho,
        sigma,
        tau,
        scenario_context,
        subflow,
        policy_context='',
        display_name=None,
        identity=None,
        free_info=None,
    ):
        return _orig_build_system_prompt(
            self,
            persona_label=persona_label,
            rho=rho,
            sigma=sigma,
            tau=tau,
            scenario_context=scenario_context,
            subflow=subflow,
            policy_context=policy_context,
            display_name=display_name,
        )

    cls.build_system_prompt = _compat_build_system_prompt
    patched_count += 1

if patched_count:
    print(f'Applied NLGLayer compatibility shim to {patched_count} class reference(s).')
else:
    print('NLGLayer already supports identity/free_info in all active class references.')

NLGLayer already supports identity/free_info in all active class references.


In [100]:
env = SupportEnv(
    artifacts_root=str(artifacts_root),
    nlg_enabled=NLG_ENABLED,
    subflow_filter=SUBFLOW_FILTER
)

obs, info = env.reset(seed=SEED)

print('=' * 65)
print('  CUSTOMER CONTEXT - READ BEFORE STARTING')
print('=' * 65)
print(f"  Subflow:        {env.state['subflow']}")
print(f"  Customer tier:  {env.state['tier']} ({env.state.get('member_level', '?')})")
print(f"  Persona:        {env.state['persona_label']}")
print()
print('  Persona parameters:')
print(f"    rho (patience):          {env.state['rho']:.3f}  {'HIGH - will stay engaged' if env.state['rho'] > 0.6 else 'LOW - may disengage quickly'}")
print(f"    sigma (sensitivity):     {env.state['sigma']:.3f}  {'HIGH - reacts strongly to failures' if env.state['sigma'] > 0.6 else 'LOW - stays calm under pressure'}")
print(f"    tau (failure tolerance): {env.state['tau']:.3f}  {'HIGH - tolerates repeated failures' if env.state['tau'] > 0.5 else 'LOW - gives up after few failures'}")
print()
difficulty_label = 'hard' if env.state['difficulty'] > 0.8 else 'medium' if env.state['difficulty'] > 0.55 else 'easy'
print(f"  Task difficulty:  {env.state['difficulty']:.3f}  ({difficulty_label})")
print()

rag_ctx = getattr(env, 'rag_context', {}) or {}
scenario_payload = rag_ctx.get('scenario', {}) if isinstance(rag_ctx, dict) else {}

if scenario_payload:
    print('  Lumo issue context:')
    print(f"    Issue type:      {rag_ctx.get('display_name', env.state['subflow'].replace('_', ' ').title())}")
    print(f"    Scenario:        {scenario_payload.get('description', 'n/a')}")
    identity = scenario_payload.get('identity', {}) if isinstance(scenario_payload.get('identity', {}), dict) else {}
    slots = scenario_payload.get('slots', {}) if isinstance(scenario_payload.get('slots', {}), dict) else {}
    if identity:
        print(f"    Customer:        {identity.get('customer_name', 'n/a')}")
    if slots:
        print(f"    Workspace:       {slots.get('workspace_name', 'n/a')} ({slots.get('workspace_url', 'n/a')})")
        print(f"    Plan:            {slots.get('plan', env.state.get('tier', 'n/a'))}")
        print(f"    Issue detail:    {slots.get('issue_detail', slots.get('error_message', 'n/a'))}")
else:
    print(f"  Scenario: {env.state['subflow'].replace('_', ' ')} (no specific details available)")
print()

initial_customer_message = None
if hasattr(env, 'slot_tracker') and env.slot_tracker is not None:
    env.slot_tracker.reveal_next(n=2)

if NLG_ENABLED and getattr(env, 'nlg_layer', None) is not None:
    active_rag_context = getattr(env, 'rag_context', {}) or {}
    scenario = active_rag_context.get('scenario', {}) if isinstance(active_rag_context, dict) else {}
    identity = scenario.get('identity', {}) if scenario else {}
    slots = scenario.get('slots', {}) if scenario else {}
    free_info = scenario.get('free_info', {}) if scenario else {}
    display_name = env.rag_context.get('display_name', env.state.get('subflow', '')) if env.rag_context else ''

    # Build a SHORT high-level topic description - NOT the full issue_detail.
    # The customer should open vaguely and let the agent draw out details.
    issue_detail = slots.get('issue_detail', '')
    workspace_name = slots.get('workspace_name', '')

    # Create a brief topic hint; avoid amounts/dates/seat counts/technical dump in opener.
    topic_hint = display_name.lower().replace('_', ' ')

    opener_prompt = f"""You are starting a customer support chat. Write your opening message.

Your problem type: {topic_hint}
Full background (DO NOT recite this - use it only as context you hold in reserve): {issue_detail}

CRITICAL RULES FOR YOUR OPENER:
- Write 1-2 sentences MAXIMUM
- State only the general nature of your problem - do NOT include specific amounts, dates, seat counts, error codes, or technical details
- Those details should only come out when the agent specifically asks for them
- Think of how a real person starts a support chat - they say 'Hi, I have a question about my billing' not 'Hi, I was charged $576 on March 1st for 32 seats at $18 per user and...'
- Do NOT list everything you know upfront
- Sound natural and human - slightly informal is fine
- If your issue involves money, just say you have a billing question - do not say the amount
- If your issue involves a technical problem, just describe the symptom briefly - do not list all troubleshooting steps you've tried

Good opener examples:
- 'Hi, I have a question about switching our billing plan.'
- 'Hey, I am having trouble logging into my account.'
- 'Hi there, we got charged something unexpected this month and I want to understand it.'
- 'I need some help - our team cannot access a feature we thought was included in our plan.'

Bad opener examples (too much detail):
- 'Hi, I am trying to figure out how our monthly Business+ plan at $18 per user for 32 users will be affected...'
- 'I just noticed a charge of $374 on March 1st for our 25-seat Business+ workspace...'"""

    scenario_context = env.state['subflow']
    if scenario_payload:
        scenario_context = json.dumps(scenario_payload, default=str)
    elif hasattr(env, 'slot_tracker') and env.slot_tracker is not None:
        scenario_context = env.slot_tracker.get_revealed_context()

    opening_utterance = env.nlg_layer.generate_utterance(
        system_prompt=env.nlg_layer.build_system_prompt(
            persona_label=env.state['persona_label'],
            rho=env.state['rho'],
            sigma=env.state['sigma'],
            tau=env.state['tau'],
            scenario_context=scenario_context,
            subflow=env.state['subflow'],
            policy_context=rag_ctx.get('policy_context', ''),
            display_name=rag_ctx.get('display_name', env.state['subflow'].replace('_', ' ').title()),
            identity=active_rag_context.get('scenario', {}).get('identity', {}) if isinstance(active_rag_context, dict) else {},
            free_info=active_rag_context.get('scenario', {}).get('free_info', {}) if isinstance(active_rag_context, dict) else {},
        ),
        conversation_history=[],
        turn_prompt=opener_prompt,
    )
    initial_customer_message = opening_utterance
    env.conversation_history.append({'role': 'assistant', 'content': opening_utterance})
    print(f"  CUSTOMER OPENER: {opening_utterance}")
else:
    fallback_issue = rag_ctx.get('display_name', env.state['subflow'].replace('_', ' '))
    print(f"  CUSTOMER OPENER: [NLG disabled - customer has {fallback_issue} issue]")

print('-' * 65)
print('  INITIAL STATE')
print('-' * 65)
fr = env.state['frustration']
fr_label = 'HIGH' if fr > 0.6 else 'MODERATE' if fr > 0.3 else 'LOW'
print(f"  Information:    {env.state['information']:.3f}")
print(f"  Progress:       {env.state['progress']:.3f}")
print(f"  Frustration:    {fr:.3f}  [{fr_label}]")
print(f"  Failed streak:  {env.state['failed_streak']}")
print(f"  p_success:      {env.state_engine.compute_p_success(env.state):.3f}")
print(f"  p_churn:        {env.reward_engine.compute_p_churn(env.state['frustration'], env.state['failed_streak'], env.state['turn_count'], env.state['tau']):.3f}")
print(f"  p_dropout:      {env.state_engine.compute_p_dropout(env.state):.3f}")
print('=' * 65)

episode_log = []
total_reward = 0.0
total_reward_tracker = [0.0]
turn = 0

print()
print('  ACTION REFERENCE:')
print('  0 = AskInfo          (request missing information)')
print('  1 = ProvideSolution  (attempt to solve the problem)')
print('  2 = AffectiveRepair  (express empathy/apology)')
print('  3 = Escalate         (route to human agent)')
print('  4 = Close            (end the conversation)')
print('=' * 65)

DocumentStore loaded:
  lumo_doc1_company_product_overview.md: 22 chunks
  lumo_doc2_support_policies.md: 48 chunks
  lumo_doc3_issue_playbooks.md: 59 chunks
  Total chunks: 129
  Mean chunk size: 787 characters
  Min chunk size: 45 characters
  Max chunk size: 1000 characters
  CUSTOMER CONTEXT - READ BEFORE STARTING
  Subflow:        status_service_added
  Customer tier:  Enterprise (gold)
  Persona:        low_engagement_resolver

  Persona parameters:
    rho (patience):          0.520  LOW - may disengage quickly
    sigma (sensitivity):     0.331  LOW - stays calm under pressure
    tau (failure tolerance): 0.616  HIGH - tolerates repeated failures

  Task difficulty:  1.000  (hard)

  Lumo issue context:
    Issue type:      New Feature Appeared
    Scenario:        AI button appeared unexpectedly after renewal
    Workspace:       Vector Labs (lumo.com/vectorlabs)
    Plan:            Enterprise
    Issue detail:    After our Business+ subscription renewed this month, a Lumo AI

In [97]:
import ipywidgets as widgets
from IPython.display import display, clear_output

if env.state.get('done', False):
    outcome_label = 'RESOLVED' if env.state.get('resolved') else 'ESCALATED' if env.state.get('escalated') else 'DROPPED OFF' if env.state.get('dropped_off') else 'TIMED OUT'
    print(f"Episode is already complete ({outcome_label}). Run Cell 3 for a new episode or Cell 5 for the transcript.")
else:
    turn += 1
    frustration = env.state['frustration']
    f_indicator = 'HIGH' if frustration > 0.6 else 'MODERATE' if frustration > 0.3 else 'LOW'

    print(f"\n{'━' * 65}")
    print(f"  TURN {turn}  |  Turn count: {env.state['turn_count']}  |  Remaining: {env.T_max - env.state['turn_count']} turns")
    print(f"{'━' * 65}")
    print(
        f"  info={env.state['information']:.3f}  "
        f"progress={env.state['progress']:.3f}  "
        f"frustration={env.state['frustration']:.3f} {f_indicator}  "
        f"streak={env.state['failed_streak']}"
    )
    print(
        f"  p_success={env.state_engine.compute_p_success(env.state):.3f}  "
        f"p_churn={env.reward_engine.compute_p_churn(env.state['frustration'], env.state['failed_streak'], env.state['turn_count'], env.state['tau']):.3f}  "
        f"p_dropout={env.state_engine.compute_p_dropout(env.state):.3f}"
    )

    if env.conversation_history:
        last_customer_msg = next(
            (m['content'] for m in reversed(env.conversation_history) if m['role'] == 'assistant'),
            None,
        )
        if last_customer_msg:
            print(f"\n  CUSTOMER: {last_customer_msg}")

    if env.state_engine.compute_p_success(env.state) > 0.6:
        print(
            f"\n  Good moment to try ProvideSolution "
            f"(p_success={env.state_engine.compute_p_success(env.state):.3f})"
        )
    if frustration > 0.6:
        print('  Frustration HIGH - consider AffectiveRepair')
    if env.state_engine.compute_p_dropout(env.state) > 0.15:
        print(f"  Dropout risk elevated ({env.state_engine.compute_p_dropout(env.state):.3f})")

    print()

    action_widget = widgets.ToggleButtons(
        options=[
            ('0: AskInfo', 0),
            ('1: ProvideSolution', 1),
            ('2: AffectiveRepair', 2),
            ('3: Escalate', 3),
            ('4: Close', 4),
        ],
        description='Action:',
        button_style='',
        tooltips=[
            'Request missing information from customer',
            'Attempt to solve the problem',
            'Express empathy or apologize',
            'Route to human agent (escalate)',
            'End the conversation',
        ],
    )

    agent_text_widget = widgets.Textarea(
        value='',
        placeholder='Type what you said to the customer (optional)...',
        description='Agent text:',
        layout=widgets.Layout(width='700px', height='80px'),
    )

    submit_button = widgets.Button(
        description='Submit Turn',
        button_style='primary',
        icon='check',
        layout=widgets.Layout(width='150px', height='40px'),
    )

    output_area = widgets.Output()

    def on_submit(_):
        action = action_widget.value
        agent_text = agent_text_widget.value.strip() or None
        action_name = env.ACTION_NAMES[action]

        with output_area:
            clear_output(wait=True)

            pre_info = env.state['information']
            pre_progress = env.state['progress']
            pre_frustration = env.state['frustration']
            pre_streak = env.state['failed_streak']

            print(f"  AGENT [{action_name}]: {agent_text or '(no text)'}")

            obs, reward, done, truncated, step_info = env.step(action, agent_text=agent_text)
            total_reward_tracker[0] += reward

            transition = step_info.get('last_transition_outcome', {})

            if NLG_ENABLED and env.conversation_history:
                last_msg = next(
                    (m['content'] for m in reversed(env.conversation_history) if m['role'] == 'assistant'),
                    None,
                )
                if last_msg:
                    print(f"\n  CUSTOMER: {last_msg}")
            elif not NLG_ENABLED:
                print('\n  CUSTOMER: [NLG disabled]')

            delta_info = env.state['information'] - pre_info
            delta_progress = env.state['progress'] - pre_progress
            delta_frustration = env.state['frustration'] - pre_frustration
            delta_streak = env.state['failed_streak'] - pre_streak

            print('\n  -- Transition Analysis --------------------------------------')

            def fmt_delta(label, pre, post, delta, good_if_positive=True):
                arrow = 'up' if delta > 0.001 else 'down' if delta < -0.001 else 'flat'
                good = (delta > 0) == good_if_positive
                marker = 'OK' if (abs(delta) < 0.001) else ('OK' if good else 'WARN')
                return f"  {label:<14} {pre:.3f} -> {post:.3f}  {arrow} {delta:+.3f} {marker}"

            print(fmt_delta('Information:', pre_info, env.state['information'], delta_info, True))
            print(fmt_delta('Progress:', pre_progress, env.state['progress'], delta_progress, True))
            print(fmt_delta('Frustration:', pre_frustration, env.state['frustration'], delta_frustration, False))
            print(f"  Failed streak:   {pre_streak} -> {env.state['failed_streak']}  delta {delta_streak:+d}")

            if action_name == 'AskInfo':
                gain = transition.get('delta_i', 0.0)
                gain_occurred = transition.get('gain_occurred', abs(gain) > 0.001)
                print(
                    f"\n  AskInfo: {'Information gained ({:.3f})'.format(gain) if gain_occurred else 'No information gained'}"
                )
            elif action_name == 'ProvideSolution':
                outcome = transition.get('outcome', 'unknown')
                p_s = transition.get('p_success_used', 0.0)
                print(f"\n  ProvideSolution: {outcome.upper()} (p_success={p_s:.3f})")
                if outcome == 'success' and env.state['progress'] >= env.state_engine.auto_resolve_threshold:
                    print(
                        f"  AUTO-RESOLVE (progress={env.state['progress']:.3f} >= {env.state_engine.auto_resolve_threshold})"
                    )
            elif action_name == 'AffectiveRepair':
                effective = transition.get('repair_effective', False)
                p_r = transition.get('p_repair_used', 0.0)
                print(f"\n  AffectiveRepair: {'EFFECTIVE' if effective else 'INEFFECTIVE'} (p_repair={p_r:.3f})")
            elif action_name == 'Close':
                cs = transition.get('close_score', 0.0)
                resolved = transition.get('resolved', False)
                print(f"\n  Close score: {cs:.3f} vs threshold {env.state_engine.close_threshold:.3f}")
                print(f"  Result: {'RESOLVED' if resolved else 'NOT RESOLVED'}")

            print(f"\n  Step reward: {reward:+.4f}  |  Episode total: {total_reward_tracker[0]:+.4f}")

            episode_log.append(
                {
                    'turn': turn,
                    'action': action_name,
                    'agent_text': agent_text,
                    'customer_utterance': next(
                        (m['content'] for m in reversed(env.conversation_history) if m['role'] == 'assistant'),
                        None,
                    )
                    if NLG_ENABLED and env.conversation_history
                    else None,
                    'state_post': {
                        'information': round(env.state['information'], 4),
                        'progress': round(env.state['progress'], 4),
                        'frustration': round(env.state['frustration'], 4),
                        'failed_streak': env.state['failed_streak'],
                    },
                    'reward': round(reward, 4),
                    'transition': {k: (round(v, 4) if isinstance(v, float) else v) for k, v in transition.items()},
                }
            )

            if done:
                terminal_type = transition.get('terminal_type', 'unknown')
                print(f"\n{'=' * 60}")
                print(f"  EPISODE ENDED - {terminal_type.upper()}")
                print(
                    f"  Outcome: {'RESOLVED' if env.state.get('resolved') else 'ESCALATED' if env.state.get('escalated') else 'DROPOUT' if env.state.get('dropped_off') else 'TIMEOUT'}"
                )
                print(f"  Total reward: {total_reward_tracker[0]:+.4f}  |  Turns: {turn}")
                print('  Run Cell 5 to see full transcript')
                print(f"{'=' * 60}")
            else:
                print('\n  Re-run this cell for the next turn')

    submit_button.on_click(on_submit)

    display(
        widgets.VBox([
            action_widget,
            agent_text_widget,
            submit_button,
            output_area,
        ])
    )


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TURN 12  |  Turn count: 13  |  Remaining: 7 turns
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  info=0.828  progress=0.476  frustration=0.023 LOW  streak=0
  p_success=0.679  p_churn=0.023  p_dropout=0.006

  CUSTOMER: Yeah, please go ahead and close the window!

  Good moment to try ProvideSolution (p_success=0.679)



In [81]:
print(f"\n{'═' * 65}")
print('  FULL CONVERSATION TRANSCRIPT')
print(f"{'═' * 65}")
print(f"  Subflow: {env.state['subflow']} | Persona: {env.state['persona_label']} | Tier: {env.state['tier']}")
outcome_label = 'RESOLVED' if env.state.get('resolved') else 'ESCALATED' if env.state.get('escalated') else 'DROPPED OFF' if env.state.get('dropped_off') else 'TIMED OUT'
print(f"  Outcome: {outcome_label}")
print(f"{'═' * 65}\n")

if 'initial_customer_message' in globals() and initial_customer_message:
    print('  Turn 0 [Customer opener]')
    print(f"  Customer: {initial_customer_message}")
    print()

for entry in episode_log:
    print(f"  Turn {entry['turn']} [{entry['action']}]")
    if entry['agent_text']:
        print(f"  Agent: {entry['agent_text']}")
    if entry['customer_utterance']:
        print(f"  Customer: {entry['customer_utterance']}")

    f_val = entry['state_post']['frustration']
    f_bar = '█' * int(f_val * 10) + '░' * (10 - int(f_val * 10))
    i_val = entry['state_post']['information']
    i_bar = '█' * int(i_val * 10) + '░' * (10 - int(i_val * 10))
    p_val = entry['state_post']['progress']
    p_bar = '█' * int(p_val * 10) + '░' * (10 - int(p_val * 10))

    print(f"  Info [{i_bar}] {i_val:.2f}  Prog [{p_bar}] {p_val:.2f}  Frust [{f_bar}] {f_val:.2f}  R={entry['reward']:+.3f}")
    print()

print(f"{'─' * 65}")
print(f"  Total reward: {total_reward_tracker[0]:+.4f}")
print(f"{'═' * 65}")

log_path = repo_root / 'Simulation_4' / 'artifacts' / 'active_sim_logs'
log_path.mkdir(parents=True, exist_ok=True)
log_file = log_path / f"episode_{env.state['subflow']}_{turn}turns.json"
with open(log_file, 'w') as f:
    json.dump({
        'subflow': env.state['subflow'],
        'tier': env.state['tier'],
        'persona': env.state['persona_label'],
        'rho': env.state['rho'],
        'sigma': env.state['sigma'],
        'tau': env.state['tau'],
        'initial_customer_message': initial_customer_message if 'initial_customer_message' in globals() else None,
        'outcome': 'resolved' if env.state.get('resolved') else 'escalated' if env.state.get('escalated') else 'dropout' if env.state.get('dropped_off') else 'timeout',
        'total_reward': total_reward_tracker[0],
        'turns': turn,
        'episode_log': episode_log
    }, f, indent=2)
print(f"\n  Episode log saved to: {log_file}")


═════════════════════════════════════════════════════════════════
  FULL CONVERSATION TRANSCRIPT
═════════════════════════════════════════════════════════════════
  Subflow: status_service_added | Persona: low_engagement_resolver | Tier: Enterprise
  Outcome: RESOLVED
═════════════════════════════════════════════════════════════════

  Turn 0 [Customer opener]
  Customer: Hey! We've got an unexpected feature showing up in all our channels and I'm hoping you can help us figure out how to turn it off.

  Turn 1 [AskInfo]
  Agent: Hello! Sure! Could you tell me more about what feature it is!
  Customer: It's an AI button that showed up in all our channels after our Business+ subscription renewed this month. The workspace name is Vector Labs.
  Info [████░░░░░░] 0.43  Prog [░░░░░░░░░░] 0.04  Frust [█░░░░░░░░░] 0.17  R=-0.100

  Turn 2 [AffectiveRepair]
  Agent: When your Business+ plan renewed, Lumo automatically activated the advanced AI features that are now bundled into Business+ — pre